<a href="https://colab.research.google.com/github/Erjg1012/Procesamiento-de-Lenguaje-Natural-PLN-/blob/main/extraccion_de_topicos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd


In [ ]:
df = pd.read_csv('datos.txt')
df

,no.reseña,title,rating,review_text,location,hotel,label
0,0,Excelente y personal amable,5,Un hotel muy bueno. El personal fue muy amabl...,Seville_Province_of_Seville_Andalucia,H10_Casa_de_la_Plata,1
1,1,Céntrico,4,"Muy buen hotel al nivel de lo esperado, habita...",Seville_Province_of_Seville_Andalucia,H10_Casa_de_la_Plata,1
2,2,Hotel excepcional,5,Magnífico hotel. La verdad es que todo perfect...,Seville_Province_of_Seville_Andalucia,H10_Casa_de_la_Plata,1
3,3,WOW!!,5,"Hotel hermoso, buen diseño, original, limpio. ...",Seville_Province_of_Seville_Andalucia,H10_Casa_de_la_Plata,1
4,4,Magnifico,5,Magnífica ubicación en pleno centro de Sevilla...,Seville_Province_of_Seville_Andalucia,H10_Casa_de_la_Plata,1
...,...,...,...,...,...,...,...
96,96,Asturias,4,La verdad es que fue una apuesta a ciegas y no...,Seville_Province_of_Seville_Andalucia,Hotel_Posada_del_Lucero,1
97,97,4 noches en sevilla,4,Para pasar 4 noches en Sevilla hemos elegido e...,Seville_Province_of_Seville_Andalucia,Hotel_Posada_del_Lucero,1
98,98,Espectacular,5,"Hotel muy cuidado y céntrico, trato muy bueno ...",Seville_Province_of_Seville_Andalucia,Hotel_Posada_del_Lucero,1
99,99,Magnífico hotel,5,Muy buena situación. Habitación muy amplia y ...,Seville_Province_of_Seville_Andalucia,Hotel_Posada_del_Lucero,1


In [ ]:
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize

In [ ]:
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
def segmentar_texto_completo(texto):
    """
    Segmenta el texto primero por oraciones (usando NLTK)
    y luego cada oración por comas.
    """
    # 1. Separa el texto en oraciones
    # Usamos language='spanish' para mejor precisión con puntuación en español
    oraciones = sent_tokenize(texto, language='spanish')

    segmentos_finales = []

    # 2. Itera sobre cada oración y sepárala por comas
    for oracion in oraciones:
        partes_coma = oracion.split(',')

        # 3. Limpia y añade cada parte
        for parte in partes_coma:
            segmento_limpio = parte.strip() # .strip() elimina espacios en blanco al inicio y final
            if segmento_limpio: # Nos aseguramos de no añadir strings vacíos
                segmentos_finales.append(segmento_limpio)

    return segmentos_finales

In [ ]:
# Aplicar la función a la columna 'review_text'
# El resultado será una nueva columna 'segmentos' que contiene una lista de strings
df['segmentos'] = df['review_text'].apply(segmentar_texto_completo)

# Imprimir el resultado
df[['review_text','segmentos']]

,review_text,segmentos
0,Un hotel muy bueno. El personal fue muy amabl...,"[Un hotel muy bueno., El personal fue muy amab..."
1,"Muy buen hotel al nivel de lo esperado, habita...","[Muy buen hotel al nivel de lo esperado, habit..."
2,Magnífico hotel. La verdad es que todo perfect...,"[Magnífico hotel., La verdad es que todo perfe..."
3,"Hotel hermoso, buen diseño, original, limpio. ...","[Hotel hermoso, buen diseño, original, limpio...."
4,Magnífica ubicación en pleno centro de Sevilla...,[Magnífica ubicación en pleno centro de Sevill...
...,...,...
96,La verdad es que fue una apuesta a ciegas y no...,[La verdad es que fue una apuesta a ciegas y n...
97,Para pasar 4 noches en Sevilla hemos elegido e...,[Para pasar 4 noches en Sevilla hemos elegido ...
98,"Hotel muy cuidado y céntrico, trato muy bueno ...","[Hotel muy cuidado y céntrico, trato muy bueno..."
99,Muy buena situación. Habitación muy amplia y ...,"[Muy buena situación., Habitación muy amplia y..."


In [ ]:
df[['review_text','segmentos']].to_excel('analisis_sentimientos_topicos.xlsx', index=False)

In [ ]:
pip install pysentimiento nltk pandas

In [ ]:
import pandas as pd
from nltk.tokenize import sent_tokenize
# 1. ESTA ES LA LÍNEA DE IMPORTACIÓN CORREGIDA
from pysentimiento import create_analyzer
import nltk


# Función de segmentación (la misma de antes)
def segmentar_texto_completo(texto):
    oraciones = sent_tokenize(texto, language='spanish')
    segmentos_finales = []
    for oracion in oraciones:
        partes_coma = oracion.split(',')
        for parte in partes_coma:
            segmento_limpio = parte.strip()
            if segmento_limpio:
                segmentos_finales.append(segmento_limpio)
    return segmentos_finales

# Aplicar la segmentación para crear las listas
df['segmentos_lista'] = df['review_text'].apply(segmentar_texto_completo)

# "Explotar" (Explode) el DataFrame
df_exploded = df.explode('segmentos_lista').reset_index(drop=True)
df_exploded = df_exploded.rename(columns={'segmentos_lista': 'segmento'})

# 2. ESTA ES LA LÍNEA DE CREACIÓN DEL ANALIZADOR CORREGIDA
# En lugar de SentimentAnalyzer(lang="es"), usamos create_analyzer
analyzer = create_analyzer(task="sentiment", lang="es")

# Función para aplicar el análisis de sentimiento (sin cambios)
def obtener_sentimiento(texto):
    if not isinstance(texto, str) or not texto:
        return "NEU"

    prediccion = analyzer.predict(texto)
    # .output nos da la etiqueta: POS, NEG, o NEU
    return prediccion.output

# Aplicar el análisis de sentimiento a la columna de segmentos (sin cambios)
df_exploded['sentimiento'] = df_exploded['segmento'].apply(obtener_sentimiento)

# Seleccionar las columnas que pediste (sin cambios)
df_final = df_exploded[['review_text', 'segmento', 'sentimiento']]

print("\n\n--- DataFrame Final (Segmentado y Analizado) ---")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/925 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/435M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]



--- DataFrame Final (Segmentado y Analizado) ---
                                           review_text  \
0    Un hotel muy bueno.  El personal fue muy amabl...   
1    Un hotel muy bueno.  El personal fue muy amabl...   
2    Un hotel muy bueno.  El personal fue muy amabl...   
3    Un hotel muy bueno.  El personal fue muy amabl...   
4    Un hotel muy bueno.  El personal fue muy amabl...   
..                                                 ...   
774  Queremos destacar la actuación rápida, eficaz ...   
775  Queremos destacar la actuación rápida, eficaz ...   
776  Queremos destacar la actuación rápida, eficaz ...   
777  Queremos destacar la actuación rápida, eficaz ...   
778  Queremos destacar la actuación rápida, eficaz ...   

                                              segmento sentimiento  
0                                  Un hotel muy bueno.         POS  
1            El personal fue muy amable y profesional.         POS  
2                 Nos gustaban desayuno mucho

In [ ]:
df_final.to_excel('datos_sentimineto.xlsx', index=False)

In [ ]:
def crear_datos_etiquetados(fila):
    """
    Crea las secuencias (X, y) para el modelo de ML
    usando el texto original y los segmentos pre-calculados.
    """
    texto_original = fila['review_text']
    segmentos_verdad = fila['segmentos']

    # 1. Tokenizar el texto original (esto será X)
    # Usamos el tokenizador de palabras de NLTK en español
    tokens_X = word_tokenize(texto_original, language='spanish')

    # 2. Obtener el primer token de cada segmento "verdadero"
    # Esto nos ayudará a saber dónde poner las etiquetas "1"
    tokens_de_inicio = set()
    for seg in segmentos_verdad:
        # Tokenizamos cada segmento y tomamos solo la primera palabra
        primer_token_segmento = word_tokenize(seg, language='spanish')
        if primer_token_segmento: # Asegurarse de que no esté vacío
            tokens_de_inicio.add(primer_token_segmento[0])

    # 3. Crear las etiquetas (esto será y)
    etiquetas_y = []
    for token in tokens_X:
        if token in tokens_de_inicio:
            etiquetas_y.append(1)
            # Importante: lo removemos para evitar colisiones si una palabra
            # se repite (ej. "Hola. Hola.")
            tokens_de_inicio.remove(token)
        else:
            etiquetas_y.append(0)

    return tokens_X, etiquetas_y

# Aplicar la nueva función para crear las columnas X, y
df[['tokens_X', 'etiquetas_y']] = df.apply(crear_datos_etiquetados, axis=1, result_type='expand')

# Imprimir el resultado para la primera fila
print("Texto Original:", df.iloc[0]['review_text'])
print("---")
print("Tokens (X):", df.iloc[0]['tokens_X'])
print("Etiquetas (y):", df.iloc[0]['etiquetas_y'])

In [ ]:
import pandas as pd
import numpy as np
from nltk.tokenize import sent_tokenize, word_tokenize
import nltk

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, TimeDistributed, Dense
from tensorflow.keras.utils import to_categorical

df = pd.read_csv('datos.txt')

# Función para obtener los segmentos "verdaderos" (tu lógica)
def segmentar_texto_completo(texto):
    oraciones = sent_tokenize(texto, language='spanish')
    segmentos_finales = []
    for oracion in oraciones:
        partes_coma = oracion.split(',')
        for parte in partes_coma:
            segmento_limpio = parte.strip()
            if segmento_limpio:
                segmentos_finales.append(segmento_limpio)
    return segmentos_finales

# Función para crear las etiquetas (X, y) para el modelo
def crear_datos_etiquetados(fila):
    texto_original = fila['review_text']
    segmentos_verdad = fila['segmentos']

    tokens_X = word_tokenize(texto_original, language='spanish')
    etiquetas_y = [0] * len(tokens_X) # Empezar con todo en 0

    # Usamos un índice para no perdernos
    token_idx_global = 0

    for seg in segmentos_verdad:
        tokens_segmento = word_tokenize(seg, language='spanish')

        if not tokens_segmento:
            continue

        # Buscar el inicio de este segmento en el texto tokenizado
        primer_token_seg = tokens_segmento[0]

        # Encontrar la próxima aparición del primer token
        for i in range(token_idx_global, len(tokens_X)):
            if tokens_X[i] == primer_token_seg:
                etiquetas_y[i] = 1 # Marcar como INICIO de segmento
                token_idx_global = i + 1 # Continuar búsqueda desde aquí
                break # Salir del bucle for interno

    return tokens_X, etiquetas_y

# Aplicar las funciones
df['segmentos'] = df['review_text'].apply(segmentar_texto_completo)
df_preparado = df.apply(crear_datos_etiquetados, axis=1, result_type='expand')
df_preparado.columns = ['tokens_X', 'etiquetas_y']

print("--- Datos Preparados (X, y) ---")
print(df_preparado)


# --- 2. VECTORIZACIÓN (Convertir texto a números) ---

# Parámetros
VOCAB_SIZE = 5000  # Tamaño máximo del vocabulario (ajustar según tus datos)
MAX_LEN = 100      # Longitud máxima de una reseña (ajustar)
EMBEDDING_DIM = 50 # Dimensión de los vectores de palabras

# Crear el Tokenizer (diccionario de palabras)
# OOV = Out-of-Vocabulary (palabra para tokens desconocidos)
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(df_preparado['tokens_X'])

# Convertir palabras a secuencias de números
X_seq = tokenizer.texts_to_sequences(df_preparado['tokens_X'])

# Aplicar padding (relleno) para que todas las secuencias midan MAX_LEN
X_pad = pad_sequences(X_seq, maxlen=MAX_LEN, padding='post', truncating='post')
y_pad = pad_sequences(df_preparado['etiquetas_y'], maxlen=MAX_LEN, padding='post', truncating='post')

# Ajustar las etiquetas 'y' para el modelo
# Necesitamos que 'y' tenga una dimensión extra para la pérdida
y_pad_expanded = np.expand_dims(y_pad, -1)

print(f"\nForma de X (entrada): {X_pad.shape}")
print(f"Forma de y (salida): {y_pad_expanded.shape}")


# --- 3. CREACIÓN DEL MODELO (Red Neuronal LSTM) ---

# Usaremos una arquitectura de "Etiquetado de Secuencias"

# Entrada: secuencias de longitud MAX_LEN
input_layer = Input(shape=(MAX_LEN,))

# Capa de Embedding: convierte números en vectores densos
# mask_zero=True le dice al LSTM que ignore el padding (0s)
embedding_layer = Embedding(input_dim=VOCAB_SIZE,
                            output_dim=EMBEDDING_DIM,
                            mask_zero=True)(input_layer)

# Capa LSTM: procesa la secuencia
# return_sequences=True es VITAL. Hace que el LSTM devuelva
# una salida para CADA token, no solo una al final.
lstm_layer = LSTM(64, return_sequences=True)(embedding_layer)

# Capa de Salida:
# TimeDistributed aplica una capa Dense a CADA paso de tiempo (cada token)
# Usamos 1 neurona con 'sigmoid' para predecir 0 o 1 (Inicio/No-Inicio)
output_layer = TimeDistributed(Dense(1, activation='sigmoid'))(lstm_layer)

# Crear el modelo
model = Model(inputs=input_layer, outputs=output_layer)

# Compilar el modelo
model.compile(optimizer='adam',
              loss='binary_crossentropy', # Perfecto para predicciones 0/1
              metrics=['accuracy'])

model.summary()


# --- 4. ENTRENAMIENTO ---

print("\n--- Iniciando Entrenamiento ---")
# (Con tan pocos datos, solo entrenamos 5 épocas como demostración)
model.fit(X_pad, y_pad_expanded, batch_size=2, epochs=10, validation_split=0.3)
print("--- Entrenamiento Completo ---")

--- Datos Preparados (X, y) ---
                                              tokens_X  \
0    [Un, hotel, muy, bueno, ., El, personal, fue, ...   
1    [Muy, buen, hotel, al, nivel, de, lo, esperado...   
2    [Magnífico, hotel, ., La, verdad, es, que, tod...   
3    [Hotel, hermoso, ,, buen, diseño, ,, original,...   
4    [Magnífica, ubicación, en, pleno, centro, de, ...   
..                                                 ...   
96   [La, verdad, es, que, fue, una, apuesta, a, ci...   
97   [Para, pasar, 4, noches, en, Sevilla, hemos, e...   
98   [Hotel, muy, cuidado, y, céntrico, ,, trato, m...   
99   [Muy, buena, situación, ., Habitación, muy, am...   
100  [Queremos, destacar, la, actuación, rápida, ,,...   

                                           etiquetas_y  
0    [1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, ...  
1    [1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, ...  
2    [1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, ...  
3    [1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 100)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 100, 50)   │    250,000 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 100)       │          0 │ input_layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 100, 64)   │     29,440 │ embedding[0][0],  │
│                     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed    │ (None, 100, 1)    │         65 │ lstm[0][0],       │
│ (TimeDistributed)   │                   │            │ not_equal[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 279,505 (1.07 MB)

 Trainable params: 279,505 (1.07 MB)

 Non-trainable params: 0 (0.00 B)


--- Iniciando Entrenamiento ---
Epoch 1/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 11s 109ms/step - accuracy: 0.8751 - loss: 0.5962 - val_accuracy: 0.9265 - val_loss: 0.3742
Epoch 2/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 2s 46ms/step - accuracy: 0.9293 - loss: 0.3553 - val_accuracy: 0.9265 - val_loss: 0.3586
Epoch 3/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 2s 46ms/step - accuracy: 0.9293 - loss: 0.3361 - val_accuracy: 0.9255 - val_loss: 0.3363
Epoch 4/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 2s 47ms/step - accuracy: 0.9275 - loss: 0.3002 - val_accuracy: 0.9232 - val_loss: 0.2891
Epoch 5/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 2s 43ms/step - accuracy: 0.9107 - loss: 0.2421 - val_accuracy: 0.9355 - val_loss: 0.1970
Epoch 6/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - accuracy: 0.9614 - loss: 0.1444 - val_accuracy: 0.9787 - val_loss: 0.1110
Epoch 7/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 3s 75ms/step - accuracy: 0.9820 - loss: 0.0808 - val_accuracy: 0.9784 - val_loss: 0.0934
Epoch 8/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 2s 47ms/step - accuracy: 0.9625 - loss